In [ ]:
!pip install sentence-transformers faiss-cpu transformers pandas pypdf

In [ ]:
# upload my law material paragraphs (relevant for questions)
from google.colab import files

uploaded = files.upload()

In [ ]:
import pandas as pd
from pypdf import PdfReader

# load cleaned dataset from csv file into dataframe
dataset = pd.read_csv("dataset_clean.csv")


# list of tax law pdf files to extract text from
pdf_files = [
    "BAO.pdf",
    "UStG1994.pdf",
    "KStG1988.pdf",
    "EStG1988.pdf"
]


# function to extract text from a single pdf file
def extract_text(pdf):
    reader = PdfReader(pdf)  # open pdf file
    text = ""
    for page in reader.pages:  # loop through all pages
        text += page.extract_text() + "\n"  # add page text to string
    return text  # return full extracted text


law_text = ""  # create empty string to store combined law text


# extract and combine text from all pdf law files
for pdf in pdf_files:
    law_text += extract_text(pdf)


# check total length of extracted legal text (number of characters)
len(law_text)

In [ ]:
# function to split long text into smaller word-based chunks
def chunk_text(text, chunk_size=500):
    words = text.split()  # split text into individual words
    chunks = []  # create empty list to store chunks

    for i in range(0, len(words), chunk_size):  # iterate in steps of chunk_size
        chunk = " ".join(words[i:i+chunk_size])  # join words into chunk
        chunks.append(chunk)  # add chunk to list

    return chunks  # return list of text chunks


# apply chunking function to extracted law text
chunks = chunk_text(law_text)

# check how many chunks were created
len(chunks)

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# load multilingual embedding model
embedder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

# convert text chunks into vector embeddings
chunk_embeddings = embedder.encode(chunks, show_progress_bar=True)

# get embedding vector size
dimension = chunk_embeddings.shape[1]

# create faiss similarity search index
index = faiss.IndexFlatL2(dimension)

# store embeddings inside index for retrieval
index.add(np.array(chunk_embeddings))

In [ ]:
from transformers import pipeline

# create text-to-text generation pipeline with flan-t5-small model
generator = pipeline(
    "text2text-generation",   # task type: instruction text generation
    model="google/flan-t5-small"  # lightweight instruction-tuned model
)

In [ ]:
def retrieve_context(question, k=2, max_chars=1200):
    # create embedding for the input question
    question_embedding = embedder.encode([question])

    # search faiss index for top-k most similar chunks
    distances, indices = index.search(
        np.array(question_embedding),
        k
    )

    context = ""

    # collect retrieved text chunks
    for i in indices[0]:
        context += chunks[i] + "\n"

    # limit context length to avoid model input overflow
    return context[:max_chars]

In [ ]:
def rag_answer(question):

    # retrieve relevant law text chunks using semantic search
    context = retrieve_context(question, k=2, max_chars=600)

    # build prompt with question + retrieved legal context
    prompt = f"""
Beantworte die steuerrechtliche Frage. Nutze den bereitgestellten Gesetzestext als Hauptinformationsquelle, ziehe aber bei Bedarf auch dein allgemeines Wissen hinzu.

Frage:
{question}

Gesetzestext:
{context}

Antwort:
"""

    # generate answer using flan-t5 model
    result = generator(
        prompt,
        max_new_tokens=40,   # limit answer length
        do_sample=False      # deterministic output
    )

    # return cleaned generated answer text
    return result[0]["generated_text"].strip()

In [ ]:
# retrieve most relevant legal text chunks for the test question
context = retrieve_context(test_question)

# print heading for readability in console output
print("\nGefundener Kontext:")

# print first 1000 characters of retrieved context (preview only)
print(context[:1000])

In [ ]:
# test
print(rag_answer(dataset.iloc[0]["prompt"]))

In [ ]:
import pandas as pd

# load dataset with questions
dataset = pd.read_csv("dataset_clean.csv")

answers = []  # list to store generated answers

total_questions = len(dataset)  # total number of questions

for i, row in dataset.iterrows():

    question_id = row["id"]        # extract question id
    question = row["prompt"]       # extract question text

    try:
        answer = rag_answer(question)  # generate rag-based answer

        # fallback if model returns empty answer
        if answer.strip() == "":
            answer = "Keine eindeutige Antwort im Gesetzestext gefunden."

    except Exception as e:
        answer = "Fehler bei der Antwortgenerierung."  # fallback if error occurs

    answers.append({
        "id": question_id,
        "answer": answer
    })

    # print progress every 50 questions
    if (i + 1) % 50 == 0:
        print(f"{i+1}/{total_questions} Fragen beantwortet")

print("fertig")  # signal completion ✅

In [ ]:
import pandas as pd

# load dataset with questions
dataset = pd.read_csv("dataset_clean.csv")

answers = []  # store generated answers
total_questions = len(dataset)

for i, row in dataset.iterrows():

    question_id = row["id"]      # question id
    question = row["prompt"]     # question text

    try:
        answer = rag_answer(question)  # generate answer via rag pipeline

        if answer.strip() == "":  # fallback if empty response
            answer = "Keine eindeutige Antwort im Gesetzestext gefunden."

    except Exception:
        answer = "Fehler bei der Antwortgenerierung."  # fallback if error

    answers.append({
        "id": question_id,
        "answer": answer
    })

    if (i + 1) % 50 == 0:  # progress update
        print(f"{i+1}/{total_questions} Fragen beantwortet")

print("fertig")

In [ ]:
# check
results_df.head()

In [ ]:
from google.colab import files
files.download("rag_model_output.csv")